# MIAC Fit Explorer

Interactive explorer for MIAC fit variants from `scripts/fits/validate_miac_fits.py`.

- Recommended backend: `ipympl`
- Install once: `pip install ipympl ipywidgets`
- Model equations render with `$...$` notation in this notebook.

In [1]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

root = Path.cwd()
for candidate in [root, *root.parents]:
    if (candidate / 'scripts' / 'fits' / 'validate_miac_fits.py').exists():
        root = candidate
        break
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

try:
    import ipympl  # noqa: F401
    get_ipython().run_line_magic('matplotlib', 'widget')
except Exception as exc:
    print('ipympl backend unavailable, falling back to inline:', exc)
    get_ipython().run_line_magic('matplotlib', 'inline')

from scripts.fits.validate_miac_fits import (
    MODEL_VARIANTS,
    discover_combos,
    plot_combo,
    resolve_variant_ids,
)

print(f'Loaded {len(MODEL_VARIANTS)} variants from validate_miac_fits.py')

Loaded 45 variants from validate_miac_fits.py


In [2]:
combos = discover_combos()
solvents = sorted({c['solvent'] for c in combos})
salts_by_solvent = {s: sorted({c['salt'] for c in combos if c['solvent'] == s}) for s in solvents}
combo_lookup = {(c['solvent'], c['salt']): c for c in combos}

profile_dd = widgets.Dropdown(
    options=['standard', '2020_variants', '2025_variants', 'all'],
    value='standard',
    description='Profile',
)
solvent_dd = widgets.Dropdown(options=solvents, value=solvents[0], description='Solvent')
salt_dd = widgets.Dropdown(options=salts_by_solvent[solvents[0]], description='Salt')
text_filter = widgets.Text(description='Filter', placeholder='variant id contains...')
refresh_btn = widgets.Button(description='Run / Refresh', button_style='primary')

def _on_solvent_change(change):
    if change['name'] != 'value':
        return
    options = salts_by_solvent[change['new']]
    salt_dd.options = options
    salt_dd.value = options[0]

solvent_dd.observe(_on_solvent_change, names='value')

controls = widgets.HBox([profile_dd, solvent_dd, salt_dd, text_filter, refresh_btn])
display(controls)

In [6]:
plot_out = widgets.Output()
checks_out = widgets.Output()
display(plot_out)
display(checks_out)

state = {'fig': None, 'lines': [], 'checkboxes': []}

def draw_plot(*_):
    with plot_out:
        plot_out.clear_output(wait=True)
        checks_out.clear_output(wait=True)

        if state['fig'] is not None:
            plt.close(state['fig'])

        combo = combo_lookup[(solvent_dd.value, salt_dd.value)]
        variant_ids = resolve_variant_ids(profile_dd.value)

        token = text_filter.value.strip().lower()
        if token:
            variant_ids = [vid for vid in variant_ids if token in vid.lower()]

        if not variant_ids:
            print('No variants matched profile/filter.')
            return

        result = plot_combo(combo, variant_ids, save=False, close=False)
        fig = result['figure']
        ax = result['axis']
        lines = ax.get_lines()

        state['fig'] = fig
        state['lines'] = lines

        print(f"Combo: {combo['salt']} in {combo['solvent']} | variants: {len(variant_ids)}")
        display(fig)

    with checks_out:
        checks_out.clear_output(wait=True)
        boxes = []
        line_offset = 1  # first line object is the scatter handle label proxy
        for line, variant_id in zip(state['lines'][line_offset:], variant_ids):
            label = MODEL_VARIANTS[variant_id]['display_label']
            cb = widgets.Checkbox(value=True, description=f"{variant_id} :: {label}")

            def _mk_handler(target_line):
                def _handler(change):
                    target_line.set_visible(change['new'])
                    target_line.figure.canvas.draw_idle()
                return _handler

            cb.observe(_mk_handler(line), names='value')
            boxes.append(cb)

        state['checkboxes'] = boxes
        display(widgets.VBox(boxes))

refresh_btn.on_click(draw_plot)
draw_plot()

Output()

Output()